<a href="https://colab.research.google.com/github/atenine/LING-230-Single-Model-Disfluency-Detector/blob/optimization/scripts/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from huggingface_hub import login
from dotenv import dotenv_values
import pandas as pd
from transformers import AutoConfig, AutoModel
import torchaudio
import torch
import torch.nn as nn
import numpy as np
from datetime import datetime
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from torch.amp import GradScaler

%load_ext tensorboard

login(dotenv_values()['HUGGINGFACE_API_KEY'])


In [ ]:
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

In [ ]:
torch.cuda.empty_cache()
import gc
gc.collect()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import tarfile

with tarfile.open('../data/SEP-28k_CLIP.zip', 'r:gz') as tar:
    tar.extractall('../data/')

clips_path = '../data/SEP-28k_CLIP'
print("Clips exist:", os.path.exists(clips_path))

In [ ]:
csv_path   = '/content/SEP-28k-Extended_clips.csv'
clips_path = '../data/SEP-28k_CLIP'
print("csv exist:", os.path.exists(csv_path))
print("Clips exist:", os.path.exists(clips_path))

In [ ]:
# Check CSV
print("CSV exists:", os.path.exists(csv_path))

# Check one audio clip
path = clips_path + '/HeStutters/10/HeStutters_10_0.wav'
print("Data Exists:", os.path.exists(path))


In [ ]:
# Load CSV
df = pd.read_csv(csv_path)

# Build filepath — Show/EpId/Show_EpId_ClipId.wav
df['filepath'] = df.apply(
    lambda row: os.path.join(
        clips_path,
        str(row['Show']),
        str(row['EpId']),
        f"{row['Show']}_{row['EpId']}_{row['ClipId']}.wav"
    ), axis=1
)

# Check how many files exist
df['exists'] = df['filepath'].apply(os.path.exists)
print(f"Total clips in CSV:     {len(df)}")
print(f"Clips found on disk:    {df['exists'].sum()}")
print(f"Clips NOT found:        {(~df['exists']).sum()}")

# Keep only existing files
df = df[df['exists']]
print("\nSample filepath:")
print(df['filepath'].iloc[0])

In [ ]:
stutter_cols = ['Prolongation', 'Block', 'SoundRep', 'WordRep', 'Interjection']
label_map={
    'Prolongation': 'prolongation',
    'Block': 'block',
    'SoundRep': 'soundrep',
    'WordRep': 'wordrep',
    'Interjection': 'interjection',
    'NoStutteredWords':'fluent'
}
cols_to_normalize = list(label_map.keys())
df[list(label_map.values())] = df[cols_to_normalize].div(3.0).rename(columns=label_map)

print("\nAverage probability per stutter type:")
for new_col in list(label_map.values())[:-1]:
    print(f"{new_col}: {df[new_col].mean():.3f}")

In [ ]:
subset_fraction = 1
train_df = df[df['SEP28k-E']== 'train'].sample(frac=subset_fraction, random_state=42).reset_index(drop=True)
valid_df = df[df['SEP28k-E']== 'dev'].sample(frac=subset_fraction, random_state=42).reset_index(drop=True)
test_df = df[df['SEP28k-E']== 'test'].sample(frac=subset_fraction, random_state=42).reset_index(drop=True)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# load voc2vec Encoder
print("\n Loading voc2vec encoder...")
# MODEL_NAME = "alkiskoudounas/voc2vec-hubert-ls-pt" # facebook/wav2vec2-base, alkiskoudounas/voc2vec-hubert-ls-pt, openai/whisper-large-v3-turbo
MODEL_NAME = "facebook/wav2vec2-base"
# MODEL_NAME = "openai/whisper-large-v3-turbo"
encoder_config = AutoConfig.from_pretrained(MODEL_NAME)
encoder = AutoModel.from_pretrained(MODEL_NAME, device_map="cuda")

In [ ]:
import soundfile as sf
# Hyperparameters
batch_size    = 32
num_epochs    = 150
learning_rate = 0.0001

class StutterDataset(torch.utils.data.Dataset):
    def __init__(self, dataframe):
        self.df = dataframe[dataframe['filepath'].apply(
            lambda f: sf.info(f).frames == 48000
        )].reset_index(drop=True)

        self.label_cols = ['prolongation', 'block', 'soundrep', 'wordrep', 'interjection']
        self.labels = torch.tensor(self.df[self.label_cols].values, dtype=torch.float32)
        print(f"Valid: {len(self.df)} | Discarded: {len(dataframe) - len(self.df)}")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        waveform, sample_rate = torchaudio.load(row['filepath'])
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        if sample_rate != 16000:
            waveform = torchaudio.functional.resample(waveform, sample_rate, 16000)
        name = f"{row['Show']}_{row['EpId']}_{row['ClipId']}"
        return waveform, self.labels[idx], name


def make_loader(df, shuffle=False):
    return DataLoader(
        StutterDataset(df),
        batch_size=batch_size,
        num_workers=4,
        pin_memory=True,
        shuffle=shuffle,
        prefetch_factor=2,
        persistent_workers=True
    )

train_loader = make_loader(train_df, shuffle=True)
val_loader   = make_loader(valid_df)
test_loader  = make_loader(test_df)

print(f"\nTrain batches: {len(train_loader)}")
print(f"Valid batches: {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")

In [ ]:
# Model definition

class AttentionPooling(nn.Module):
    def __init__(self, hidden_size):
        super(AttentionPooling, self).__init__()
        # Attention layer
        self.attention = nn.Linear(hidden_size, 1)

    def forward(self, x):
        scores  = self.attention(x)                    # → (batch, time_steps, 1)
        weights = torch.softmax(scores, dim=1)         # → (batch, time_steps, 1)
        pooled  = torch.sum(x * weights, dim=1)        # → (batch, hidden_size)
        return pooled


class StutterNet(nn.Module):
    def __init__(self, encoder_type='wav2vec'):
        super(StutterNet, self).__init__()

        if encoder_type == 'voc2vec':
            self.encoder = AutoModel.from_pretrained("alkiskoudounas/voc2vec-hubert-ls-pt", device_map="cuda")
            for param in self.encoder.parameters():
                param.requires_grad = False

        if encoder_type == 'wav2vec':
            self.encoder = AutoModel.from_pretrained('facebook/wav2vec2-base', device_map="cuda")
            for param in self.encoder.parameters():
                param.requires_grad = False

        # Layer 1 - Linear Projection
        # Reduces voc2vec output from 768 → 384
        self.projection = nn.Linear(768, 384)

        # Layer 2 - 1D CNN
        self.cnn = nn.Sequential(
            nn.Conv1d(in_channels=384, out_channels=128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Dropout(0.3),
        )

        # Layer 3 - 2 layers of BiLSTM
        self.bilstm = nn.LSTM(
            input_size=128,
            hidden_size=256,
            num_layers=2,
            batch_first=True,
            dropout=0.3,
            bidirectional=True
        )

        # Layer 4 - Attention Pooling
        # Focuses on most important parts of audio
        self.attention = AttentionPooling(hidden_size=512)

        # Layer 5 - Dense Layers
        self.dense = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
        )

        # Layer 6 - 5 Output Heads
        # One head per stutter type!
        self.head_prolongation = nn.Linear(128, 1)
        self.head_block        = nn.Linear(128, 1)
        self.head_soundrep     = nn.Linear(128, 1)
        self.head_wordrep      = nn.Linear(128, 1)
        self.head_interjection = nn.Linear(128, 1)

    def forward(self, x):
        # x shape → (batch, samples) raw audio waveform

        x = torch.squeeze(x)

        #  STEP 1 — Encoder
        with torch.no_grad():
            x = self.encoder(x).last_hidden_state

        # STEP 2 — Linear Projection
        x = self.projection(x)             # → (batch, time_steps, 384)
        x = torch.relu(x)

        # STEP 3 — 1D CNN
        x = x.permute(0, 2, 1)            # → (batch, 384, time_steps)
        x = self.cnn(x)                    # → (batch, 128, time_steps)
        x = x.permute(0, 2, 1)            # → (batch, time_steps, 128)

        # STEP 4 — BiLSTM
        x, _ = self.bilstm(x)             # → (batch, time_steps, 512)

        # STEP 5 — Attention Pooling
        x = self.attention(x)             # → (batch, 512)

        # STEP 6 — Dense Layers
        x = self.dense(x)                 # → (batch, 128)

        return {
            'prolongation': self.head_prolongation(x).squeeze(),
            'block':        self.head_block(x).squeeze(),
            'soundrep':     self.head_soundrep(x).squeeze(),
            'wordrep':      self.head_wordrep(x).squeeze(),
            'interjection': self.head_interjection(x).squeeze()
        }


# Initialize Model
encoder_name = 'wav2vec'
model  = StutterNet(encoder_type=encoder_name).to(device)
# print(model)

In [ ]:
from torch.utils.tensorboard import SummaryWriter
writer    = SummaryWriter(comment="encoder " + encoder_name)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
scaler    = GradScaler(device=device)

eval_accu   = []
eval_losses = []
classes     = ['prolongation', 'block', 'soundrep', 'wordrep', 'interjection']


def train(epoch):
    model.train()
    running_loss = {h: 0 for h in classes}
    correct      = {h: 0 for h in classes}
    total        = {h: 0 for h in classes}

    for data in tqdm(train_loader, desc="Training"):
        inputs = data[0].to(device, non_blocking=True)
        targets = {h: data[1][:, i].to(device, non_blocking=True)
                   for i, h in enumerate(classes)}

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast():
            outputs = model(inputs)
            loss = 0
            for h in classes:
                head_loss = criterion(outputs[h], targets[h])
                running_loss[h] += head_loss.item()
                loss += head_loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        for h in classes:
            predicted = torch.round(outputs[h])
            total[h]   += targets[h].size(0)
            correct[h] += predicted.eq(targets[h]).sum().item()

    for h in classes:
        writer.add_scalar(f"{h} Loss/train",     running_loss[h] / len(train_loader), epoch)
        writer.add_scalar(f"{h} Accuracy/train", 100. * correct[h] / total[h],        epoch)


def eval(epoch):
    model.eval()
    running_loss = {h: 0 for h in classes}
    correct      = {h: 0 for h in classes}
    total        = {h: 0 for h in classes}

    with torch.no_grad():
        for data in tqdm(val_loader, desc="Validating"):
            inputs  = data[0].to(device, non_blocking=True)
            targets = {h: data[1][:, i].to(device, non_blocking=True)
                       for i, h in enumerate(classes)}

            with torch.cuda.amp.autocast():
                outputs = model(inputs)
                for h in classes:
                    head_loss = criterion(outputs[h], targets[h])
                    running_loss[h] += head_loss.item()

            for h in classes:
                predicted  = torch.round(outputs[h])
                total[h]   += targets[h].size(0)
                correct[h] += predicted.eq(targets[h]).sum().item()

    for h in classes:
        validate_loss = running_loss[h] / len(val_loader)
        accu          = 100. * correct[h] / total[h]

        writer.add_scalar(f"{h} Loss/eval",     validate_loss, epoch)
        writer.add_scalar(f"{h} Accuracy/eval", accu,          epoch)

        eval_accu.append(accu)
        eval_losses.append(validate_loss)

    return validate_loss

In [ ]:
from datetime import datetime
import os

best_val_loss = float('inf')
train_start_time = datetime.now()

for epoch in tqdm(range(1, num_epochs + 1), desc="Epoch"):
    train(epoch)
    eval_loss = eval(epoch)

    # Save best model only (instead of overwriting every epoch)
    if eval_loss < best_val_loss:
        best_val_loss = eval_loss
        torch.save({
            'epoch':      epoch,
            'model':      model.state_dict(),
            'optimizer':  optimizer.state_dict(),
            'val_loss':   best_val_loss,
        }, 'DisfluencyDetector_best.pth')
        print(f"  ✓ Saved best model (val_loss: {best_val_loss:.4f})")

    torch.cuda.empty_cache()

train_end_time = datetime.now()
print(f"\nFinished training in {train_end_time - train_start_time}")
print(f"Best val loss: {best_val_loss:.4f}")

writer.close()  # flush tensorboard logs

In [ ]:
model.eval()

classToOutputMap = {
    'block':        'Block',
    'soundrep':     'SoundRep',
    'wordrep':      'WordRep',
    'interjection': 'Interjection',
    'prolongation': 'Prolongation'
}

# Tracking
total             = {h: 0 for h in classes}
correct           = {h: 0 for h in classes}
running_loss      = {h: 0 for h in classes}
predicted_stutter = {h: 0 for h in classes}
labels_stutter    = {h: 0 for h in classes}
correct_stutter   = {h: 0 for h in classes}

exportColumns = {'Clip': []}
for h in classes:
    exportColumns[classToOutputMap[h]] = []

with torch.no_grad():
    for data in tqdm(test_loader, desc="Testing"):
        inputs     = data[0].to(device, non_blocking=True)
        clip_names = data[2]
        targets    = {h: data[1][:, i].to(device, non_blocking=True)
                      for i, h in enumerate(classes)}

        with torch.cuda.amp.autocast():
            outputs = model(inputs)

        exportColumns['Clip'] += list(clip_names)

        for h in classes:
            head_loss        = criterion(outputs[h], targets[h])
            running_loss[h] += head_loss.item()

            # Apply sigmoid since model outputs logits
            probs            = torch.sigmoid(outputs[h])
            predicted_labels = torch.round(probs)

            total[h]   += targets[h].size(0)
            correct[h] += predicted_labels.eq(targets[h]).sum().item()

            # Vectorized — replaces per-element for loop
            predicted_stutter[h] += predicted_labels.sum().item()
            labels_stutter[h]    += targets[h].sum().item()
            correct_stutter[h]   += (predicted_labels * targets[h]).sum().item()

            exportColumns[classToOutputMap[h]] += probs.tolist()

# Export
exportDf = pd.DataFrame(exportColumns)
exportDf.to_csv('results.csv', index=False)
print("Saved results.csv")

# Metrics
print("\n" + "="*50)
for h in classes:
    accu      = 100. * correct[h] / total[h]
    test_loss = running_loss[h] / len(test_loader)
    precision = correct_stutter[h] / predicted_stutter[h] if predicted_stutter[h] > 0 else 0
    recall    = correct_stutter[h] / labels_stutter[h]    if labels_stutter[h]    > 0 else 0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    writer.add_scalar(f"{h} Loss/test",     test_loss, 0)
    writer.add_scalar(f"{h} Accuracy/test", accu,      0)

    print(f"\n{h} {'─'*40}")
    print(f"  Accuracy  : {accu:.2f}%")
    print(f"  Precision : {precision:.4f}")
    print(f"  Recall    : {recall:.4f}")
    print(f"  F1 Score  : {f1:.4f}")
    print(f"  Test Loss : {test_loss:.4f}")

writer.close()

In [ ]:
exportColumns['Block']